# JalanTanggap — Fase 1 End-to-End (Colab)

Pipeline: **LAKSA Excel → cleaning → NLP baseline → map matching → SBERT deduplikasi → rekap per ruas → survey → SAW → Top 10**.

Dataset sintetis hanya untuk proof-of-concept. `Hastag` tidak digunakan sebagai input klasifikasi agar tidak terjadi label leakage.

In [ ]:
!pip -q install -U openpyxl geopandas shapely sentence-transformers scikit-learn

import math, json, zipfile
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path
from shapely.geometry import Point
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from google.colab import files

REPO=Path("/content/jalantanggap")
if not REPO.exists():
    !git clone -q https://github.com/kaivanriz/jalantanggap.git /content/jalantanggap
DATA=REPO/"data"/"sample-tangerang"
LAKSA_FILE=DATA/"laporan_laksa_sintetis_fase1.xlsx"

if not LAKSA_FILE.exists():
    print("Fixture Excel belum ada di repo. Upload master LAKSA:")
    up=files.upload()
    xlsx=[x for x in up if x.lower().endswith(".xlsx")]
    if not xlsx: raise ValueError("Upload file .xlsx")
    LAKSA_FILE=Path("/content")/xlsx[0]

print("Input:", LAKSA_FILE)

## 1. Baca dan bersihkan master LAKSA

In [ ]:
raw=pd.read_excel(LAKSA_FILE,sheet_name="Pengaduan",header=None,engine="openpyxl")
cols=["No","ID Pengaduan","Dari","Tgl.Laporan","Hastag","Tujuan","Isi Pengaduan","Lokasi","Lat","Lng","Status Lapor","Tgl Proses","Waktu Respon","Tgl Selesai","Waktu TL","Sumber","Jenis Aduan","Jawaban Tanggal","Jawaban"]
df=raw.iloc[7:,:19].copy(); df.columns=cols
df=df[df["ID Pengaduan"].notna()].reset_index(drop=True)

def dec(v):
    if pd.isna(v): return np.nan
    try: return float(str(v).strip().replace(",","."))
    except: return np.nan

df["Lat"]=df["Lat"].map(dec); df["Lng"]=df["Lng"].map(dec)
df["Tgl.Laporan"]=pd.to_datetime(df["Tgl.Laporan"],errors="coerce")
for c in ["Isi Pengaduan","Lokasi","Hastag","Tujuan","Sumber"]:
    df[c]=df[c].fillna("").astype(str).str.strip()
df["text_nlp"]=(df["Isi Pengaduan"]+" Lokasi: "+df["Lokasi"]).str.replace(r"\s+"," ",regex=True).str.strip()
print("Pengaduan:",len(df))
df.head(3)

## 2. NLP baseline + map matching

In [ ]:
MODEL="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model=SentenceTransformer(MODEL)
proto={
"jalan_rusak":"jalan rusak berlubang lubang aspal retak mengelupas amblas bergelombang badan jalan",
"drainase":"drainase saluran air selokan gorong gorong genangan pembangunan saluran",
"pju":"lampu penerangan jalan umum pju mati gelap tiang lampu",
"utilitas_tiang":"tiang utilitas kabel PLN telekom telkom tiang miring",
"pohon":"pohon tumbang rawan tumbang dahan ranting",
"trotoar":"trotoar rusak pedestrian pejalan kaki paving"
}
labels=list(proto)
pv=model.encode(list(proto.values()),normalize_embeddings=True)
tv=model.encode(df["text_nlp"].tolist(),normalize_embeddings=True,show_progress_bar=True)
score=np.matmul(tv,pv.T); bi=score.argmax(axis=1)
df["kategori_prediksi"]=[labels[i] for i in bi]
df["confidence_kategori"]=score.max(axis=1).round(4)
df["road_related"]=df["kategori_prediksi"].eq("jalan_rusak")

roads=gpd.read_file(DATA/"jaringan.geojson")
if roads.crs is None: roads=roads.set_crs("EPSG:4326")
roads_m=roads.to_crs("EPSG:32748")

def nearest(r):
    if pd.isna(r.Lat) or pd.isna(r.Lng): return pd.Series([None,np.nan])
    p=gpd.GeoSeries([Point(r.Lng,r.Lat)],crs="EPSG:4326").to_crs("EPSG:32748").iloc[0]
    d=roads_m.geometry.distance(p); i=d.idxmin()
    return pd.Series([roads_m.loc[i,"id_segmen"],float(d.loc[i])])

df[["id_segmen","distance_to_road_m"]]=df.apply(nearest,axis=1)
df["map_match_valid"]=df["id_segmen"].notna() & df["distance_to_road_m"].le(50)
df[["ID Pengaduan","kategori_prediksi","id_segmen","distance_to_road_m"]].head()

## 3. SBERT deduplikasi + rekap per ruas

In [ ]:
def hav(a,b,c,d):
    R=6371000
    p1,p2=math.radians(a),math.radians(c)
    dp=math.radians(c-a); dl=math.radians(d-b)
    x=math.sin(dp/2)**2+math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2*R*math.asin(math.sqrt(x))

df["duplicate_group"]=""; df["is_duplicate"]=False
for sid,g in df[df.road_related & df.map_match_valid & df["Tgl.Laporan"].notna()].groupby("id_segmen"):
    ids=g.index.tolist(); parent={i:i for i in ids}
    def find(x):
        while parent[x]!=x:
            parent[x]=parent[parent[x]]; x=parent[x]
        return x
    def union(a,b):
        a,b=find(a),find(b)
        if a!=b: parent[b]=a
    sm=cosine_similarity(tv[ids])
    for a in range(len(ids)):
        for b in range(a+1,len(ids)):
            i,j=ids[a],ids[b]
            dist=hav(df.loc[i,"Lat"],df.loc[i,"Lng"],df.loc[j,"Lat"],df.loc[j,"Lng"])
            days=abs((df.loc[i,"Tgl.Laporan"]-df.loc[j,"Tgl.Laporan"]).total_seconds())/86400
            if sm[a,b]>=0.62 and dist<=80 and days<=30: union(i,j)
    groups={}
    for i in ids: groups.setdefault(find(i),[]).append(i)
    for n,members in enumerate(groups.values(),1):
        gid=f"DG-{sid}-{n:03d}"
        members=sorted(members,key=lambda x:df.loc[x,"Tgl.Laporan"])
        for k,i in enumerate(members):
            df.loc[i,"duplicate_group"]=gid; df.loc[i,"is_duplicate"]=k>0

df["status_pengolahan"]=np.select(
    [~df.road_related,~df.map_match_valid,df.is_duplicate],
    ["excluded_non_road","needs_location_review","duplicate"],default="valid")

hasil=df[["ID Pengaduan","Tgl.Laporan","Hastag","Isi Pengaduan","Lokasi","Lat","Lng","kategori_prediksi","confidence_kategori","road_related","id_segmen","distance_to_road_m","duplicate_group","is_duplicate","status_pengolahan"]].copy()
hasil.to_csv("/content/hasil_pengolahan_pengaduan_fase1.csv",index=False)

rr=df[df.road_related & df.map_match_valid].copy()
rekap=rr.groupby("id_segmen").agg(
    laporan_masuk=("ID Pengaduan","count"),
    laporan_duplikat=("is_duplicate","sum"),
    jumlah_laporan_valid=("duplicate_group",lambda s:s[s.ne("")].nunique())
).reset_index()
rekap=rekap.merge(roads[["id_segmen","nama"]].drop_duplicates(),on="id_segmen",how="left").rename(columns={"nama":"nama_jalan"})
rekap.to_csv("/content/rekap_pengaduan_per_ruas.csv",index=False)
rekap.sort_values("jumlah_laporan_valid",ascending=False).head(20)

## 4. SAW → rekomendasi Top 10

In [ ]:
survey=pd.read_csv(DATA/"survey_fase1.csv")
rules=json.load(open(DATA/"aturan_fase1.json",encoding="utf-8"))
x=survey.merge(rekap,on="id_segmen",how="left")
for c in ["jumlah_laporan_valid","laporan_masuk","laporan_duplikat"]:
    x[c]=x[c].fillna(0)
weighted=[]
for c in rules["saw"]["criteria"]:
    n=c["name"]; v=pd.to_numeric(x[n],errors="coerce").fillna(0).astype(float)
    if c["type"]=="benefit":
        norm=v/v.max() if v.max()>0 else 0
    else:
        p=v[v>0]; mn=p.min() if len(p) else 0
        norm=(mn/v.replace(0,np.nan)).fillna(0)
    weighted.append(norm*float(c["weight"]))
x["saw_score"]=sum(weighted)
x=x.sort_values("saw_score",ascending=False).reset_index(drop=True)
x["rank"]=np.arange(1,len(x)+1); x["saw_score_pct"]=(x.saw_score*100).round(2)
x["rekomendasi"]=np.where(x["rank"]<=10,"PRIORITAS_TOP_10","CADANGAN")
out=x[["rank","id_segmen","nama_jalan","saw_score_pct","tingkat_kerusakan","dampak_akses","peran_jalan","panjang_rusak_m","jumlah_laporan_valid","laporan_masuk","laporan_duplikat","fasilitas_kritis","lama_tidak_ditangani_hari","rekomendasi"]]
out.to_csv("/content/hasil_prioritas_jalan_fase1.csv",index=False)
out.head(10)

## 5. Download output

In [ ]:
z="/content/jalantanggap_fase1_output.zip"
with zipfile.ZipFile(z,"w",zipfile.ZIP_DEFLATED) as f:
    for p in ["hasil_pengolahan_pengaduan_fase1.csv","rekap_pengaduan_per_ruas.csv","hasil_prioritas_jalan_fase1.csv"]:
        f.write("/content/"+p,p)
files.download(z)